# HAR Complete Training Pipeline
Runs preprocessing and training in one go

In [9]:
import sys
import os
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# Add training script to path
SCRIPT_DIR = Path.cwd()
sys.path.insert(0, str(SCRIPT_DIR))

# Import training components
from har_training import (
    MLPClassifier, scale_features, one_hot_encode, 
    create_batches, compute_confusion_matrix, train,
    SEED, LEARNING_RATE, EPOCHS, BATCH_SIZE, VALIDATION_SPLIT,
    ACTIVITIES, NUM_CLASSES, INPUT_SIZE
)

## Step 1: Data Preprocessing
Load and prepare the UCI HAR dataset

In [10]:
def preprocess_data(train_path: str, test_path: str):
    """Load and preprocess UIH HAR data"""
    print("Step 1: Preprocessing Data")
    print("-" * 60)
    
    # Load raw data
    print("Loading data...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Extract features
    feature_columns = [col for col in train_df.columns if col not in ['Activity', 'subject']]
    X_train = train_df[feature_columns].values
    y_train_labels = train_df['Activity'].values
    X_test = test_df[feature_columns].values
    y_test_labels = test_df['Activity'].values
    
    # Convert activity labels to integer indices
    # Create mapping from activity name to index
    activity_to_idx = {activity: idx for idx, activity in enumerate(ACTIVITIES)}
    y_train = np.array([activity_to_idx[label] for label in y_train_labels], dtype=np.int64)
    y_test = np.array([activity_to_idx[label] for label in y_test_labels], dtype=np.int64)
    
    print(f"✓ Data loaded: {X_train.shape[0]} training, {X_test.shape[0]} test")
    print(f"✓ Features: {X_train.shape[1]}")
    print(f"✓ Activities mapped to indices: {activity_to_idx}")
    
    return X_train, y_train, X_test, y_test, feature_columns

## Step 2: Model Training
Train the MLP neural network

In [11]:
def train_model(X_train, y_train, X_test, y_test, scaling_params, output_dir: str):
    """Train MLP model"""
    print("")
    print("Step 2: Training Neural Network")
    print("-" * 60)
    
    # One-hot encode
    y_train_oh = one_hot_encode(y_train)
    y_test_oh = one_hot_encode(y_test)
    
    # Split into train/validation
    val_size = int(X_train.shape[0] * VALIDATION_SPLIT)
    indices = np.random.permutation(X_train.shape[0])
    
    X_val = X_train[indices[:val_size]]
    y_val = y_train_oh[indices[:val_size]]
    X_train_final = X_train[indices[val_size:]]
    y_train_final = y_train_oh[indices[val_size:]]
    
    print(f"Data split: {X_train_final.shape[0]} train, {X_val.shape[0]} val, {X_test.shape[0]} test")
    print("")
    
    # Train model
    model = MLPClassifier()
    history, best_weights = train(model, X_train_final, y_train_final, X_val, y_val)
    
    # Evaluate
    print("")
    print("Step 3: Evaluation")
    print("-" * 60)
    
    test_acc, test_loss = model.evaluate(X_test, y_test_oh)
    y_pred, _ = model.predict(X_test)
    cm = compute_confusion_matrix(y_test, y_pred)
    
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
    print("")
    
    # Save outputs
    print("Step 4: Saving Outputs")
    print("-" * 60)
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Save weights
    model_data = {
        'metadata': {
            'architecture': 'MLP',
            'input_size': INPUT_SIZE,
            'hidden_layers': [256, 128, 64],
            'output_size': NUM_CLASSES,
            'activations': ['relu', 'relu', 'relu', 'softmax']
        },
        'scaling_params': scaling_params,
        'class_labels': ACTIVITIES,
        'weights': best_weights
    }
    
    weights_path = f"{output_dir}/weights.json"
    with open(weights_path, 'w') as f:
        json.dump(model_data, f, indent=2)
    print(f"✓ Weights saved: {weights_path}")
    
    # Save confusion matrix
    cm_path = f"{output_dir}/confusion_matrix.json"
    with open(cm_path, 'w') as f:
        json.dump(cm.tolist(), f, indent=2)
    print(f"✓ Confusion matrix saved: {cm_path}")
    
    # Save training history
    history_path = f"{output_dir}/training_history.json"
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"✓ Training history saved: {history_path}")

## Execute Complete Training Pipeline
Run all steps: preprocessing → scaling → training → saving

In [12]:
print("")
print("=" * 60)
print("HAR Complete Training Pipeline")
print("=" * 60)
print("")

# File paths
train_path = "train.csv"
test_path = "test.csv"
output_dir = "../../public/model"

# Preprocess
X_train, y_train, X_test, y_test, feature_columns = preprocess_data(
    train_path, test_path
)

# Scale
X_train_scaled, X_test_scaled, scaling_params = scale_features(X_train, X_test)

# Train
train_model(
    X_train_scaled, y_train, X_test_scaled, y_test,
    scaling_params, output_dir
)

print("")
print("=" * 60)
print("✓ Training Pipeline Complete!")
print("=" * 60)
print("")
print("Web app is ready. Run: npm run dev")
print("")


HAR Complete Training Pipeline

Step 1: Preprocessing Data
------------------------------------------------------------
Loading data...
✓ Data loaded: 7352 training, 2947 test
✓ Features: 561
✓ Activities mapped to indices: {'WALKING': 0, 'WALKING_UPSTAIRS': 1, 'WALKING_DOWNSTAIRS': 2, 'SITTING': 3, 'STANDING': 4, 'LAYING': 5}
Scaling parameters (from training set only):
  Data min: -1.000000
  Data max: 1.000000

Step 2: Training Neural Network
------------------------------------------------------------
Data split: 6250 train, 1102 val, 2947 test


Starting training for 100 epochs...
Learning rate: 0.001, Batch size: 32
------------------------------------------------------------
Epoch   1/100 | Train Loss: 0.1429 | Val Loss: 0.1584 | Train Acc: 0.9470 | Val Acc: 0.9419
Epoch   5/100 | Train Loss: 0.1717 | Val Loss: 0.1967 | Train Acc: 0.9234 | Val Acc: 0.9129
Epoch  10/100 | Train Loss: 0.0569 | Val Loss: 0.0640 | Train Acc: 0.9765 | Val Acc: 0.9728
Epoch  15/100 | Train Loss: 0.06